In [26]:
import pytest

@pytest.mark.example
def test_example():
    assert 1 + 1 == 2
    assert "a" == "a"

test_example()

In [8]:
from openai import OpenAI
import dotenv
import os
from pydantic import BaseModel
from typing import Literal

dotenv.load_dotenv("../.env")

client = OpenAI(
    base_url=os.environ.get("ENDPOINT"),
    api_key=os.environ.get("TOKEN")
)

class TypeHeroe(BaseModel):
    type: Literal["Strength", "Agility", "Intelligence"]
    response: str



def _resolve_type_heroe(message:str) -> TypeHeroe:
    type_response = client.responses.parse(
        model="gpt-4.1",
        input=[
            {
                "role":"system",
                "content":"You are a hero type resolver. You will receive a message and you will"
            },
            {
                "role": "user",
                "content": message
            }
        ],
        text_format=TypeHeroe,
        max_output_tokens=100,
        temperature=0
    )
    return type_response.output_parsed

In [22]:
def test_resolve_type_heroe():
    result = _resolve_type_heroe("I am a strong warrior with great strength.")
    assert result.type == "Strength"
    assert len(result.response) > 5
    assert isinstance(result.response, str)
    
    result = _resolve_type_heroe("I am agile and quick on my feet.")
    assert result.type == "Agility"
    assert isinstance(result.response, str)
    
    result = _resolve_type_heroe("I am a wise mage with great intelligence.")
    assert result.type == "Intelligence"
    assert isinstance(result.response, str)


In [16]:
def test_resolve_type_heroe_semantic():
    result = _resolve_type_heroe("Sniper")
    assert result.type == "Strength"
    assert isinstance(result.response, str)
    
    result = _resolve_type_heroe("me gusta instalar minas")
    assert result.type == "Agility"
    assert isinstance(result.response, str)
    
    result = _resolve_type_heroe("Soy un mago sabio con gran inteligencia.")
    assert result.type == "Intelligence"
    assert isinstance(result.response, str)

In [23]:
def run_tests():
    tests = [
        test_resolve_type_heroe,
        test_resolve_type_heroe_semantic
    ]

    passed = 0

    for test in tests:
        try:
            test()
            print(f"✅ Test {test.__name__} passed.")
            passed += 1
        except AssertionError as e:
            print(f"❌Test {test.__name__}")
    
    print(f"\n{passed}/{len(tests)} tests passed.")

run_tests()

✅ Test test_resolve_type_heroe passed.
❌Test test_resolve_type_heroe_semantic

1/2 tests passed.
